In [1]:
"""
BreastCare Kenya — Clinical Decision Support Platform
Streamlit App | 6 Modules:
  1. Quick Risk Assessment
  2. Screening Checklist
  3. Smart Follow-Up Tracker
  4. Referral Intelligence
  5. Analytics Dashboard
  6. AI Clinical Assistant (smart offline engine — no API key needed)

Run:
    pip install streamlit plotly pandas openpyxl
    streamlit run breast_cancer_kap_app.py
"""

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import random
import time

# ── Page config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="BreastCare Kenya",
    page_icon="🎗️",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Global CSS ────────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@300;400;600;700;800&display=swap');

html, body, [class*="css"], .stMarkdown, .stText,
button, input, select, textarea, label, p, h1, h2, h3, h4, h5 {
    font-family: 'Nunito', sans-serif !important;
}

/* ── Sidebar ─────────────────────────────────────────── */
section[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #1a0a2e 0%, #2d1054 60%, #4a1068 100%);
    border-right: 1px solid rgba(255,255,255,0.08);
}
section[data-testid="stSidebar"] * {
    color: #f0e6ff !important;
    font-family: 'Nunito', sans-serif !important;
}
section[data-testid="stSidebar"] .stRadio label {
    background: rgba(255,255,255,0.06);
    border-radius: 10px;
    padding: 10px 14px;
    margin: 3px 0;
    display: block;
    transition: background 0.2s;
    cursor: pointer;
    font-weight: 600;
    font-size: .93rem;
}
section[data-testid="stSidebar"] .stRadio label:hover {
    background: rgba(255,255,255,0.14);
}

/* ── Main background ─────────────────────────────────── */
.main { background: #faf7ff; }

/* ── Cards ───────────────────────────────────────────── */
.kcard {
    background: white;
    border-radius: 16px;
    padding: 1.4rem 1.6rem;
    box-shadow: 0 2px 16px rgba(74,16,104,0.08);
    margin-bottom: 1rem;
    border-left: 5px solid #9c27b0;
    font-family: 'Nunito', sans-serif;
}
.kcard.red    { border-color: #e53935; background: #fff5f5; }
.kcard.orange { border-color: #f57c00; background: #fff8f0; }
.kcard.green  { border-color: #2e7d32; background: #f5fff5; }
.kcard.blue   { border-color: #1565c0; background: #f0f6ff; }
.kcard.purple { border-color: #6a1b9a; }
.kcard h3 { margin: 0 0 .3rem; font-size: 1.1rem; font-weight: 700; }
.kcard p  { margin: 0; color: #555; font-size: .9rem; }

/* ── AI Assistant card ───────────────────────────────── */
.ai-card {
    background: linear-gradient(135deg, #1a0a2e 0%, #2d1054 100%);
    border-radius: 16px;
    padding: 1.4rem 1.6rem;
    margin-bottom: 1rem;
    border: 1px solid rgba(192,132,252,0.3);
    font-family: 'Nunito', sans-serif;
    box-shadow: 0 4px 24px rgba(74,16,104,0.2);
}
.ai-card h3 { margin: 0 0 .3rem; font-size: 1.1rem; font-weight: 700; color: #e9d5ff; }
.ai-card p  { margin: 0; color: #c4b5fd; font-size: .9rem; }

.ai-response {
    background: white;
    border-radius: 12px;
    padding: 1.2rem 1.5rem;
    border-left: 4px solid #7c3aed;
    box-shadow: 0 2px 12px rgba(124,58,237,0.1);
    font-size: .95rem;
    line-height: 1.7;
    color: #2d1054;
    font-family: 'Nunito', sans-serif;
    margin-top: .75rem;
}

.chat-user {
    background: #f3e8ff;
    border-radius: 12px 12px 4px 12px;
    padding: .75rem 1rem;
    margin: .5rem 0 .5rem auto;
    max-width: 80%;
    font-size: .9rem;
    color: #2d1054;
    font-weight: 600;
    text-align: right;
}
.chat-ai {
    background: white;
    border-radius: 12px 12px 12px 4px;
    padding: .75rem 1rem;
    margin: .5rem auto .5rem 0;
    max-width: 85%;
    font-size: .9rem;
    color: #1a0a2e;
    border: 1px solid #e9d5ff;
    box-shadow: 0 1px 6px rgba(124,58,237,0.08);
    line-height: 1.65;
}

/* ── Risk badge ──────────────────────────────────────── */
.risk-badge {
    display: inline-block; padding: 7px 20px;
    border-radius: 30px; font-weight: 800;
    font-size: 1rem; margin: 10px 0;
    letter-spacing: .04em; text-transform: uppercase;
    font-family: 'Nunito', sans-serif;
}
.risk-HIGH     { background: #e53935; color: white; }
.risk-MODERATE { background: #f57c00; color: white; }
.risk-LOW      { background: #2e7d32; color: white; }

/* ── Section headers ─────────────────────────────────── */
.section-head {
    font-size: 1.75rem; font-weight: 800;
    color: #2d1054;
    border-bottom: 3px solid #9c27b0;
    padding-bottom: .4rem; margin-bottom: 1.1rem;
    font-family: 'Nunito', sans-serif;
}
.section-sub {
    color: #6a1b9a; font-weight: 600;
    font-size: .93rem; margin-bottom: 1rem;
    font-family: 'Nunito', sans-serif;
}

/* ── Referral cards ──────────────────────────────────── */
.ref-card {
    border-radius: 12px; padding: 1rem 1.4rem;
    font-weight: 700; font-size: .97rem;
    text-align: center; margin: .5rem 0;
    font-family: 'Nunito', sans-serif;
}
.ref-URGENT    { background: #e53935; color: white; }
.ref-ROUTINE   { background: #f57c00; color: white; }
.ref-IMAGING   { background: #1565c0; color: white; }
.ref-EDUCATION { background: #2e7d32; color: white; }

/* ── Metric tiles ────────────────────────────────────── */
.mtile {
    background: white; border-radius: 14px;
    padding: 1.2rem; text-align: center;
    box-shadow: 0 2px 10px rgba(74,16,104,0.1);
    border-top: 4px solid #9c27b0;
    font-family: 'Nunito', sans-serif;
}
.mtile h2 { font-size: 2.1rem; margin: 0; color: #2d1054; font-weight: 800; }
.mtile p  { margin: 0; color: #888; font-size: .85rem; font-weight: 600; }

/* ── Check items ─────────────────────────────────────── */
.check-item {
    background: white; border-radius: 10px;
    padding: .75rem 1rem; margin: .35rem 0;
    box-shadow: 0 1px 6px rgba(0,0,0,0.05);
    border-left: 4px solid #9c27b0;
    font-size: .91rem; font-family: 'Nunito', sans-serif;
}
.check-item.flag { border-color: #e53935; background: #fff5f5; font-weight: 700; }

/* ── Story banner ────────────────────────────────────── */
.story-banner {
    background: linear-gradient(90deg, #880e4f, #4a148c);
    border-radius: 14px; padding: 1.1rem 1.5rem;
    color: white; margin-bottom: 1.1rem;
    font-family: 'Nunito', sans-serif;
}
.story-banner h4 { margin: 0 0 .3rem; font-size: 1.05rem; font-weight: 800; }
.story-banner p  { margin: 0; font-size: .87rem; opacity: .9; }

/* ── Connectivity badge — matches purple sidebar ──────── */
.conn-badge {
    display: inline-block; border-radius: 20px;
    padding: 5px 16px; font-size: .78rem;
    font-weight: 700; font-family: 'Nunito', sans-serif;
    letter-spacing: .03em;
}
.conn-online  { background: rgba(192,132,252,.15); color: #d8b4fe; border: 1.5px solid #9d5cd0; }
.conn-offline { background: rgba(167,139,250,.10); color: #c4b5fd; border: 1.5px solid #7c3aed; }
</style>
""", unsafe_allow_html=True)


# ── Session state ─────────────────────────────────────────────────────────────
if "patients" not in st.session_state:
    st.session_state.patients = [
        {"id":"PT-001","name":"Grace Wanjiku","age":42,"county":"Nairobi",
         "facility":"Kenyatta National Hospital","practitioner":"Nurse Achieng",
         "risk":"HIGH","referral":"Urgent",
         "next_due":(datetime.today()-timedelta(days=3)).strftime("%Y-%m-%d"),
         "status":"Overdue","notes":"Palpable lump, nipple discharge"},
        {"id":"PT-002","name":"Fatuma Hassan","age":35,"county":"Mombasa",
         "facility":"Coast General Hospital","practitioner":"Dr. Mwenda",
         "risk":"MODERATE","referral":"Imaging",
         "next_due":(datetime.today()+timedelta(days=2)).strftime("%Y-%m-%d"),
         "status":"Pending","notes":"Family history positive"},
        {"id":"PT-003","name":"Esther Chebet","age":28,"county":"Uasin Gishu",
         "facility":"Moi Teaching Hospital","practitioner":"Clinical Officer Ruto",
         "risk":"LOW","referral":"Education",
         "next_due":(datetime.today()+timedelta(days=30)).strftime("%Y-%m-%d"),
         "status":"Scheduled","notes":"Routine screening"},
        {"id":"PT-004","name":"Mary Auma","age":55,"county":"Kisumu",
         "facility":"Kisumu County Hospital","practitioner":"Nurse Otieno",
         "risk":"HIGH","referral":"Urgent",
         "next_due":(datetime.today()-timedelta(days=7)).strftime("%Y-%m-%d"),
         "status":"Overdue","notes":"Post-menopausal bleeding, skin changes"},
        {"id":"PT-005","name":"Jane Muthoni","age":48,"county":"Kiambu",
         "facility":"Thika Level 5","practitioner":"Dr. Kamau",
         "risk":"MODERATE","referral":"Routine",
         "next_due":(datetime.today()+timedelta(days=14)).strftime("%Y-%m-%d"),
         "status":"Pending","notes":"Asymmetric density on self-exam"},
        {"id":"PT-006","name":"Rose Njeri","age":31,"county":"Nakuru",
         "facility":"Nakuru County Referral","practitioner":"Midwife Wambui",
         "risk":"LOW","referral":"Education",
         "next_due":(datetime.today()-timedelta(days=1)).strftime("%Y-%m-%d"),
         "status":"Overdue","notes":"First visit, no symptoms"},
    ]

if "screenings" not in st.session_state:
    counties    = ["Nairobi","Mombasa","Kisumu","Kiambu","Nakuru","Uasin Gishu","Homa Bay"]
    facilities  = ["Public Hospital","Health Centre","Private Hospital","Dispensary","Community"]
    professions = ["Nurse","Clinical Officer","Doctor/Medical Officer","Midwife","Community Health Worker"]
    random.seed(42)
    rows = []
    for _ in range(120):
        d = datetime.today() - timedelta(days=random.randint(0, 89))
        rows.append({
            "date":       d.strftime("%Y-%m-%d"),
            "month":      d.strftime("%b %Y"),
            "county":     random.choice(counties),
            "facility":   random.choice(facilities),
            "profession": random.choice(professions),
            "risk":       random.choices(["HIGH","MODERATE","LOW"],   weights=[15,35,50])[0],
            "referral":   random.choices(["Urgent","Imaging","Routine","Education"], weights=[12,20,30,38])[0],
            "followed_up":random.choices([True, False], weights=[55,45])[0],
        })
    st.session_state.screenings = pd.DataFrame(rows)

if "grace_demo"              not in st.session_state: st.session_state.grace_demo              = False
if "checklist_step"          not in st.session_state: st.session_state.checklist_step          = 1
if "ai_chat_history"         not in st.session_state: st.session_state.ai_chat_history         = []
if "last_assessment_context" not in st.session_state: st.session_state.last_assessment_context = None


# ══════════════════════════════════════════════════════════════════════════════
# OFFLINE AI ENGINE — smart template logic, zero API calls
# ══════════════════════════════════════════════════════════════════════════════

def generate_clinical_summary(ctx: dict) -> str:
    name     = ctx["name"]
    age      = ctx["age"]
    risk     = ctx["risk"]
    score    = ctx["score"]
    referral = ctx["referral"]
    symptoms = ctx["symptoms"]
    rf       = ctx["risk_factors"]
    facility = ctx["facility"]
    county   = ctx["county"]
    action   = ctx["ref_action"]

    urgency_map = {
        "HIGH":     ("requires urgent clinical attention", "same-day", "immediately"),
        "MODERATE": ("warrants prompt investigation",      "within 2 weeks", "soon"),
        "LOW":      ("is currently low risk",              "at annual review", "at the next scheduled visit"),
    }
    urgency_phrase, timeframe, adverb = urgency_map[risk]

    symptom_line = (
        f"presenting with {symptoms}"
        if symptoms != "No acute red flags"
        else "with no acute presenting symptoms"
    )
    rf_line = (
        f"Background risk factors include {rf}."
        if rf != "none documented"
        else "No significant background risk factors are documented."
    )

    referral_line = {
        "Urgent":    f"She has been triaged for an URGENT referral and should be seen by an oncology or surgical unit today. Do not send her home without this referral being actioned.",
        "Imaging":   f"A mammogram or breast ultrasound has been requested and should be completed within 14 days. Ensure the imaging request is given to the patient before she leaves.",
        "Routine":   f"A routine follow-up appointment has been scheduled. Educate the patient on breast self-examination before discharge.",
        "Education": f"No clinical referral is required at this time. Provide breast self-examination (BSE) education and schedule an annual screening.",
    }.get(referral, action)

    return (
        f"**Clinical Handover — {name}, Age {age} | {facility}, {county}**\n\n"
        f"{name} is a {age}-year-old patient {symptom_line}. "
        f"Her risk score is **{score}/100**, placing her in the **{risk} RISK** category — "
        f"this {urgency_phrase}. "
        f"{rf_line} "
        f"{referral_line} "
        f"\n\n*Next action due: {timeframe}. Practitioner to confirm patient has understood and accepted the care plan.*"
    )


def generate_referral_letter(ctx: dict) -> str:
    today_str = datetime.today().strftime("%d %B %Y")
    name      = ctx["name"]
    age       = ctx["age"]
    symptoms  = ctx["symptoms"]
    risk      = ctx["risk"]
    facility  = ctx["facility"]
    county    = ctx["county"]
    rf        = ctx["risk_factors"]

    urgency_label = {
        "HIGH": "URGENT — SAME DAY",
        "MODERATE": "SEMI-URGENT — WITHIN 2 WEEKS",
        "LOW": "ROUTINE",
    }.get(risk, "ROUTINE")

    return (
        f"**REFERRAL LETTER**\n\n"
        f"**Date:** {today_str}\n"
        f"**From:** {facility}, {county} County\n"
        f"**To:** Oncology / Breast Surgery Unit\n"
        f"**Priority:** {urgency_label}\n\n"
        f"---\n\n"
        f"Dear Colleague,\n\n"
        f"I am referring **{name}**, a **{age}-year-old** patient, for specialist review of suspected breast pathology. "
        f"The patient presented to this facility with {symptoms if symptoms != 'No acute red flags' else 'symptoms requiring further assessment'}. "
        f"{'Background risk factors include: ' + rf + '. ' if rf != 'none documented' else ''}"
        f"Clinical assessment using the BreastCare Kenya risk scoring tool returned a **{risk} RISK** result, "
        f"indicating that further specialist evaluation is warranted {urgency_label.lower()}.\n\n"
        f"Please review and advise on further management. All documentation is available on request.\n\n"
        f"Yours faithfully,\n"
        f"[Practitioner Name]\n"
        f"[Designation] — {facility}\n"
        f"[Phone / NHIF No.]"
    )


BSE_SWAHILI = """**Jinsi ya Kujifanyia Uchunguzi wa Matiti Nyumbani (BSE)**
*Fanya hivi kila mwezi, siku 7–10 baada ya hedhi yako*

---

**Hatua ya 1 — Angalia Mbele ya Kioo 👁️**
Simama mbele ya kioo, mikono upande. Tazama matiti yote mawili. Angalia kama kuna tofauti ya umbo, rangi ya ngozi, au chuchu.

**Hatua ya 2 — Inua Mikono Juu 🙌**
Inua mikono juu ya kichwa, endelea kutazama. Angalia kama kuna msukosuko wa ngozi au chuchu inavyovutwa ndani.

**Hatua ya 3 — Palpation — Lala Chini 🛏️**
Lala chini, weka mto chini ya bega la kulia. Tumia vidole vitatu vya mkono wa kushoto kupapasa titi la kulia kwa mzunguko mdogo mdogo. Fuata mstari kutoka juu hadi chini. Rudia upande wa pili.

**Hatua ya 4 — Angalia Chuchu 🔍**
Kwa upole bonyeza chuchu kuelekea ndani. Kama kuna maji yanayotoka (si maziwa), au maumivu, nenda hospitali haraka.

**Hatua ya 5 — Palpation Ukiwa Umesimama 🚿**
Unaweza kufanya hivi ukiwa kwenye bafu, ngozi ikiwa na maji. Tumia mkono mmoja kupapasa titi lingine.

---

⚠️ **Nenda Hospitali Mara Moja Ukiona:**
- Uvimbe (lump) wowote mpya
- Ngozi inayokauka au kuwa kama ngozi ya machungwa
- Chuchu inavyovutwa ndani
- Maumivu yanayoendelea bila sababu

*Uchunguzi huu hausaidii kuthibitisha saratani — unaenda hospitali ukigundua kitu chochote kipya.*"""


def ai_knowledge_base(user_msg: str, ctx: dict | None) -> str:
    """
    Route user questions to pre-written, Kenya-specific clinical responses.
    Matches on keywords. Falls back to a helpful general answer.
    """
    msg = user_msg.lower()

    # ── Quick-action routes ───────────────────────────────────────────────────
    if "summarise" in msg and "patient" in msg:
        if ctx:
            return generate_clinical_summary(ctx)
        return "No patient has been assessed yet in this session. Please go to **🩺 Risk Assessment**, run a case, then come back here."

    if "referral letter" in msg or "draft referral" in msg or "referral" in msg and "letter" in msg:
        if ctx:
            return generate_referral_letter(ctx)
        return (
            "No patient is loaded yet. Please complete a risk assessment first, "
            "then click **📝 Draft referral letter** again and I'll generate a personalised letter."
        )

    if "swahili" in msg or "kiswahili" in msg or "bse" in msg and "swahili" in msg:
        return BSE_SWAHILI

    if "analytics" in msg or "statistics" in msg or "screening data" in msg or "compliance" in msg:
        df_s = st.session_state.screenings
        n    = len(df_s)
        hi   = len(df_s[df_s["risk"] == "HIGH"])
        fu   = round(df_s["followed_up"].mean() * 100, 1)
        pct_hi = round(hi / n * 100, 1)
        top_county = df_s["county"].value_counts().idxmax()
        low_fu_county = (
            df_s.groupby("county")["followed_up"].mean()
            .mul(100).round(1).idxmin()
        )
        concern = "below the 60% WHO minimum threshold" if fu < 60 else "meeting the WHO minimum threshold"
        return (
            f"**Analytics Interpretation for County Health Officer**\n\n"
            f"Over the last 90 days, **{n} screenings** have been recorded across facilities. "
            f"Of these, **{hi} ({pct_hi}%) are classified as HIGH RISK** — "
            f"each of these patients requires urgent action if not already referred.\n\n"
            f"**Follow-up compliance is currently {fu}%**, which is {concern}. "
            f"The county with the lowest compliance rate is **{low_fu_county}** — "
            f"this should be a priority for the next supervisory visit.\n\n"
            f"**{top_county}** has the highest screening volume, suggesting either stronger demand or better reporting infrastructure.\n\n"
            f"**Recommended actions:**\n"
            f"1. Contact practitioners in {low_fu_county} to understand follow-up barriers\n"
            f"2. Review all HIGH RISK cases to confirm referral completion\n"
            f"3. Consider a community health worker outreach campaign in lower-performing counties\n"
            f"4. Export the CSV and share with the County Director of Health monthly"
        )

    # ── Clinical knowledge questions ──────────────────────────────────────────
    if "peau d'orange" in msg or "peau d" in msg:
        return (
            "**Peau d'orange** (French for 'orange peel') describes a skin change where the breast skin "
            "develops a dimpled, pitted texture — like the surface of an orange. "
            "It is caused by tumour cells blocking the lymphatic vessels under the skin, causing swelling "
            "and tethering the skin at the hair follicle points.\n\n"
            "**Clinical significance:** This is a **red flag sign** and is associated with locally advanced "
            "or inflammatory breast cancer. Any patient presenting with this should be referred URGENTLY "
            "to a surgical or oncology unit — do not wait for imaging results before referring."
        )

    if "nipple discharge" in msg or "discharge" in msg and "nipple" in msg:
        return (
            "**Nipple Discharge — What to Watch For**\n\n"
            "Not all nipple discharge is concerning. Here is how to classify it:\n\n"
            "🟢 **Likely benign** — Milky discharge in a woman who has recently breastfed; "
            "discharge only when squeezed; bilateral and from multiple ducts.\n\n"
            "🟡 **Needs investigation** — Clear or yellow discharge; unilateral; persistent.\n\n"
            "🔴 **Red flag — refer urgently** — **Bloody or blood-stained discharge**; "
            "spontaneous (without squeezing); unilateral from a single duct; "
            "associated with a palpable lump.\n\n"
            "In Kenya, the most common causes of pathological discharge are intraductal papilloma "
            "and breast carcinoma. Any bloody or spontaneous discharge should trigger an urgent referral."
        )

    if "bse" in msg or "self exam" in msg or "self-exam" in msg or "breast self" in msg:
        return (
            "**Breast Self-Examination (BSE) — 5 Steps**\n\n"
            "Advise patients to perform BSE monthly, 7–10 days after their period starts "
            "(when breasts are least tender). Post-menopausal women should pick a fixed date each month.\n\n"
            "**1. Visual inspection** — Stand in front of a mirror with arms at sides. "
            "Look for changes in size, shape, skin, or nipple position. Repeat with arms raised.\n\n"
            "**2. Manual examination (lying down)** — Place a folded towel under the right shoulder. "
            "Using the left hand, use three fingers in small circular motions, covering the whole breast "
            "from armpit to sternum. Apply light, medium, and firm pressure.\n\n"
            "**3. Nipple check** — Gently squeeze each nipple. Note any discharge.\n\n"
            "**4. Axillary check** — Feel the armpit area for any lumps or swelling.\n\n"
            "**5. Repeat standing** — Can be done in the shower.\n\n"
            "Teach patients to report: any new lump, skin change, nipple inversion, "
            "bloody discharge, or persistent pain."
        )

    if "family history" in msg or "brca" in msg or "genetic" in msg:
        return (
            "**Family History & Genetic Risk in Breast Cancer**\n\n"
            "Family history is one of the strongest risk factors. Here is how to classify it:\n\n"
            "🟡 **Moderate risk** — One first-degree relative (mother, sister, daughter) diagnosed "
            "with breast or ovarian cancer, especially under age 50.\n\n"
            "🔴 **High risk** — Two or more first- or second-degree relatives; "
            "a male relative with breast cancer; known BRCA1 or BRCA2 mutation in the family; "
            "a relative with bilateral breast cancer.\n\n"
            "**In the Kenyan context:** Genetic testing (BRCA) is not widely available in public facilities. "
            "However, a strong family history alone is sufficient to escalate a patient's risk category "
            "and lower the threshold for referral. "
            "Document all first- and second-degree family history at every screening visit."
        )

    if "referral" in msg and ("pathway" in msg or "where" in msg or "which hospital" in msg or "send" in msg):
        return (
            "**Kenya Breast Cancer Referral Pathways**\n\n"
            "Referrals follow the county health system tier structure:\n\n"
            "**Level 2–3 (Dispensary / Health Centre):** Can perform initial screening and risk assessment. "
            "Refer to Level 4 for imaging or Level 5/6 for specialist care.\n\n"
            "**Level 4 (Sub-county Hospital):** Can order mammography or ultrasound. "
            "Refer biopsy-required or HIGH RISK cases upward.\n\n"
            "**Level 5–6 (County / National Referral):**\n"
            "- **Kenyatta National Hospital (KNH)** — Nairobi; has oncology unit\n"
            "- **Moi Teaching & Referral Hospital** — Eldoret; covers Rift Valley\n"
            "- **Coast General Hospital** — Mombasa; covers Coast region\n"
            "- **Kisumu County Referral** — covers Nyanza\n\n"
            "**NHIF coverage:** Breast cancer treatment is covered under NHIF's chronic illness package. "
            "Advise patients to bring their NHIF card to every appointment. "
            "For those without cover, contact the county social welfare office."
        )

    if "stage" in msg or "staging" in msg or "early" in msg and "late" in msg:
        return (
            "**Breast Cancer Staging — Why It Matters in Kenya**\n\n"
            "Kenya has one of the highest rates of late-stage breast cancer presentation in sub-Saharan Africa. "
            "KAP studies (2013–2024) show that 60–80% of patients present at Stage III or IV, "
            "when 5-year survival drops below 30%.\n\n"
            "**Stage overview:**\n"
            "- **Stage I–II (Early):** Tumour confined to breast ± a few nodes. 5-year survival: 70–99%\n"
            "- **Stage III (Locally advanced):** Skin/chest wall involvement, multiple nodes. Survival: 40–70%\n"
            "- **Stage IV (Metastatic):** Spread to other organs. Survival: ~20–30%\n\n"
            "**This tool exists because** structured screening and immediate referral at Stage I–II "
            "is the single most effective way to improve outcomes at a population level. "
            "Every patient you refer today could be the one you save."
        )

    if "who" in msg and ("guideline" in msg or "recommendation" in msg or "standard" in msg):
        return (
            "**WHO / IARC Breast Cancer Screening Guidelines (Low-Resource Settings)**\n\n"
            "For countries like Kenya with limited mammography infrastructure, WHO recommends:\n\n"
            "**1. Clinical Breast Examination (CBE)** as the primary screening tool — "
            "this is what BreastCare Kenya is built around.\n\n"
            "**2. Target age:** Women aged 40–69 as priority; screen from 25 if high-risk.\n\n"
            "**3. Frequency:** Every 1–2 years for average risk; annually for high risk.\n\n"
            "**4. CBE by trained health workers** (nurses, clinical officers, CHWs) is considered "
            "effective and feasible in settings where radiological screening is not available.\n\n"
            "**5. Mammography** remains the gold standard but requires infrastructure; "
            "ultrasound is preferred for women under 40 or with dense breasts.\n\n"
            "Source: WHO Guide to Cancer Early Diagnosis (2017) & IARC Handbooks Vol. 15"
        )

    if "menopause" in msg or "post-menopausal" in msg or "hormonal" in msg or "hrt" in msg:
        return (
            "**Menopause & Hormone Use in Breast Cancer Risk**\n\n"
            "Post-menopausal status is an independent breast cancer risk factor. Here is why:\n\n"
            "- Oestrogen exposure over a lifetime drives many breast cancers. "
            "Later menopause = more years of exposure = higher risk.\n"
            "- **Post-menopausal women with any new breast lump must be referred urgently** — "
            "benign lumps are far less common after menopause.\n\n"
            "**Hormone Replacement Therapy (HRT):** Combined oestrogen-progestogen HRT increases "
            "breast cancer risk by approximately 26% (Million Women Study). "
            "Risk begins within 1–2 years of use and decreases after stopping.\n\n"
            "**Oral Contraceptive Pills (OCP):** Modest increase in risk during use, "
            "which returns to baseline within 10 years of stopping.\n\n"
            "**In practice:** Document all HRT/OCP use and duration. "
            "Any post-menopausal patient on HRT presenting with a breast complaint should be referred."
        )

    # ── Generic fallback ──────────────────────────────────────────────────────
    topics = [
        "peau d'orange", "nipple discharge", "BSE / breast self-examination",
        "family history and BRCA", "referral pathways in Kenya",
        "staging and why early detection matters", "WHO/IARC screening guidelines",
        "menopause and hormone therapy",
    ]
    topics_list = "\n".join([f"- {t}" for t in topics])
    return (
        f"I can answer clinical questions about breast cancer screening, risk factors, "
        f"referral pathways, and patient communication in the Kenyan context.\n\n"
        f"**Try asking me about:**\n{topics_list}\n\n"
        f"You can also use the **Quick Actions** above to:\n"
        f"- Summarise the last patient assessed\n"
        f"- Draft a personalised referral letter\n"
        f"- Get BSE instructions in Swahili\n"
        f"- Get an analytics interpretation for a health officer"
    )


# ── Sidebar ───────────────────────────────────────────────────────────────────
st.sidebar.markdown("""
<div style='text-align:center;padding:1.1rem 0 .6rem;'>
  <div style='font-size:2.2rem;'>🎗️</div>
  <div style='font-family:Nunito,sans-serif;font-size:1.3rem;font-weight:800;
              color:#f0e6ff;letter-spacing:.02em;'>BreastCare Kenya</div>
  <div style='font-size:.78rem;color:#c9a8e8;margin-top:.25rem;font-weight:600;'>
    Clinical Decision Support
  </div>
</div>
<hr style='border-color:rgba(255,255,255,0.12);margin:.5rem 0 1rem;'>
""", unsafe_allow_html=True)

module = st.sidebar.radio("", [
    "🩺  Risk Assessment",
    "✅  Screening Checklist",
    "🔔  Follow-Up Tracker",
    "🔀  Referral Intelligence",
    "📊  Analytics Dashboard",
    "🤖  AI Clinical Assistant",
])

st.sidebar.markdown("<hr style='border-color:rgba(255,255,255,0.12);margin:1rem 0;'>", unsafe_allow_html=True)

# ── Connectivity badge — purple palette to match sidebar ──────────────────────
try:
    import urllib.request
    urllib.request.urlopen("https://dns.google", timeout=2)
    conn_class, conn_text = "conn-online", "● Online"
except Exception:
    conn_class, conn_text = "conn-offline", "● Offline — cached data active"

st.sidebar.markdown(
    f"<div style='text-align:center;'>"
    f"<span class='conn-badge {conn_class}'>{conn_text}</span></div>",
    unsafe_allow_html=True
)

st.sidebar.markdown("""
<hr style='border-color:rgba(255,255,255,0.12);margin:1rem 0;'>
<div style='font-size:.75rem;color:#c9a8e8;text-align:center;line-height:1.7;font-family:Nunito,sans-serif;'>
  Calibrated from Kenya KAP literature<br>2013–2024 · Synthetic demo data<br>
  <em>Not a substitute for clinical judgement</em>
</div>
""", unsafe_allow_html=True)


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 1 — RISK ASSESSMENT
# ══════════════════════════════════════════════════════════════════════════════
if "Risk Assessment" in module:
    st.markdown('<div class="section-head">🩺 Quick Breast Cancer Risk Assessment</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Complete the form to calculate patient risk level and recommended action</div>', unsafe_allow_html=True)

    with st.expander("🎬 Hackathon Demo — Follow Grace's Journey", expanded=False):
        st.markdown("""
        <div class="story-banner">
          <div>
            <h4>👩🏾 Meet Grace Wanjiku — 42, Nairobi</h4>
            <p>A mother of three who almost ignored a lump she found three months ago.
            Watch how BreastCare Kenya catches what could have been missed.</p>
          </div>
        </div>
        """, unsafe_allow_html=True)
        st.markdown("""
        **Without this app:** Grace visited a busy health centre. She mentioned a lump almost as an afterthought.
        The nurse had no structured tool. Grace was told *"come back if it gets worse."*
        Three months later she returned — Stage III.

        **With BreastCare Kenya:** That first visit flags HIGH RISK in under 2 minutes,
        triggers an urgent referral, and automatically tracks her follow-up.
        """)
        col_a, col_b = st.columns(2)
        if col_a.button("🚀 Pre-fill Grace's Case", use_container_width=True):
            st.session_state.grace_demo = True
            st.rerun()
        if col_b.button("🔄 Clear / New Patient", use_container_width=True):
            st.session_state.grace_demo = False
            st.rerun()

    grace = st.session_state.grace_demo
    if grace:
        st.info("👩🏾 **Grace's case loaded.** Review the pre-filled form below and click Calculate Risk.")

    with st.form("risk_form"):
        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown("**Patient Information**")
            p_name       = st.text_input("Patient Name",      value="Grace Wanjiku" if grace else "", placeholder="e.g. Grace Wanjiku")
            p_age        = st.number_input("Age (years)", 18, 90, value=42 if grace else 40)
            p_county     = st.selectbox("County", ["Nairobi","Mombasa","Kisumu","Kiambu","Nakuru","Uasin Gishu","Homa Bay","Other"])
            p_facility   = st.selectbox("Facility", ["Public Hospital","Health Centre","Private Hospital","Dispensary","Community"], index=1 if grace else 0)
            practitioner = st.text_input("Practitioner Name", value="Nurse Achieng" if grace else "", placeholder="Your name")
        with c2:
            st.markdown("**Symptoms & Signs**")
            lump          = st.checkbox("Palpable breast lump",                   value=True  if grace else False)
            nipple_dc     = st.checkbox("Nipple discharge (non-milk)",            value=True  if grace else False)
            skin_changes  = st.checkbox("Skin changes (peau d'orange, dimpling)", value=False)
            nipple_invert = st.checkbox("Nipple inversion / retraction",          value=False)
            axillary      = st.checkbox("Axillary lymph node enlargement",        value=True  if grace else False)
            breast_pain   = st.checkbox("Persistent unexplained breast pain",     value=True  if grace else False)
            ulceration    = st.checkbox("Ulceration / open wound on breast",      value=False)
        with c3:
            st.markdown("**Risk Factors**")
            fam_hist    = st.selectbox("Family history of breast/ovarian cancer",
                                       ["None","1st-degree relative","2+ relatives or BRCA known"],
                                       index=1 if grace else 0)
            menarche    = st.selectbox("Age at first period",  ["≥13 years (normal)","<13 years (early)"])
            menopause   = st.selectbox("Menopausal status",    ["Pre-menopausal","Post-menopausal (<5 yrs)","Post-menopausal (>5 yrs)"])
            parity      = st.selectbox("Parity",               ["Parous, first child <30","Nulliparous or first child ≥30"])
            breastfed   = st.checkbox("Breastfed for ≥1 year (protective)", value=True  if grace else False)
            hrt         = st.checkbox("Current HRT / OCP use (>5 yrs)",     value=False)
            prev_biopsy = st.checkbox("Previous abnormal breast biopsy",    value=False)
            obesity     = st.checkbox("BMI ≥30 / obesity",                  value=False)

        submitted = st.form_submit_button("🔍  Calculate Risk & Recommend Action", use_container_width=True)

    if submitted:
        score = 0
        if ulceration:    score += 40
        if axillary:      score += 30
        if skin_changes:  score += 25
        if nipple_invert: score += 20
        if lump:          score += 20
        if nipple_dc:     score += 15
        if breast_pain:   score += 5
        if fam_hist == "2+ relatives or BRCA known": score += 25
        elif fam_hist == "1st-degree relative":      score += 15
        if menopause == "Post-menopausal (>5 yrs)":  score += 15
        elif menopause == "Post-menopausal (<5 yrs)":score += 8
        if menarche == "<13 years (early)":          score += 8
        if parity == "Nulliparous or first child ≥30":score += 8
        if hrt:          score += 8
        if prev_biopsy:  score += 10
        if obesity:      score += 5
        if breastfed:    score -= 8
        if p_age >= 50:   score += 15
        elif p_age >= 40: score += 8
        elif p_age >= 35: score += 4

        if score >= 50 or ulceration or (lump and axillary) or (lump and skin_changes):
            risk = "HIGH"
        elif score >= 25:
            risk = "MODERATE"
        else:
            risk = "LOW"

        if risk == "HIGH":
            referral   = "Urgent"
            ref_action = "Refer IMMEDIATELY to oncology/surgical unit. Do not delay. Document and call ahead."
            ref_class  = "red"
        elif risk == "MODERATE":
            if lump or axillary:
                referral   = "Imaging"
                ref_action = "Order mammogram/ultrasound within 2 weeks. Follow up with patient directly."
            else:
                referral   = "Routine"
                ref_action = "Schedule follow-up within 4–6 weeks. Educate on BSE. Review at next visit."
            ref_class = "orange"
        else:
            referral   = "Education"
            ref_action = "Provide breast self-examination (BSE) education. Schedule annual screening."
            ref_class  = "green"

        red_flags = [s for s, v in {
            "Palpable lump": lump, "Nipple discharge": nipple_dc,
            "Skin changes": skin_changes, "Nipple inversion": nipple_invert,
            "Axillary nodes": axillary, "Ulceration": ulceration,
        }.items() if v]

        next_due = (datetime.today() + timedelta(
            days=1 if risk == "HIGH" else 14 if risk == "MODERATE" else 365
        )).strftime("%Y-%m-%d")

        st.markdown("---")
        st.markdown("### 📋 Assessment Result")

        rc1, rc2 = st.columns([1, 2])
        with rc1:
            score_colour = "#e53935" if risk=="HIGH" else "#f57c00" if risk=="MODERATE" else "#2e7d32"
            ref_icon     = "🚨" if referral=="Urgent" else "🖼️" if referral=="Imaging" else "📅" if referral=="Routine" else "📚"
            st.markdown(f"""
            <div class="kcard {ref_class}">
              <h3>{p_name or 'Patient'} · Age {p_age}</h3>
              <p>{p_county} · {p_facility}</p><br>
              <div style='font-size:.82rem;color:#888;font-weight:600;'>Risk Score</div>
              <div style='font-size:2.6rem;font-weight:800;color:{score_colour};line-height:1.1;'>
                {min(score, 100)}
              </div>
              <span class='risk-badge risk-{risk}'>{risk} RISK</span><br><br>
              <div style='font-size:.82rem;color:#888;font-weight:600;'>Referral Decision</div>
              <div class='ref-card ref-{referral}' style='margin-top:.4rem;'>
                {ref_icon} {referral} Referral
              </div>
            </div>
            """, unsafe_allow_html=True)

        with rc2:
            st.markdown(f"""
            <div class="kcard blue">
              <h3>Recommended Action</h3>
              <p style='font-size:1rem;color:#1a237e;font-weight:700;'>{ref_action}</p>
            </div>
            """, unsafe_allow_html=True)

            if red_flags:
                flags_html = "".join([f"<li style='margin:.3rem 0;'>⚠️ {f}</li>" for f in red_flags])
                st.markdown(f"""
                <div class="kcard red">
                  <h3>🚩 Red Flags Detected</h3>
                  <ul style='margin:.5rem 0 0;padding-left:1.2rem;'>{flags_html}</ul>
                </div>
                """, unsafe_allow_html=True)

            st.markdown(f"""
            <div class="kcard purple">
              <h3>📅 Follow-Up Due</h3>
              <p style='font-size:1rem;font-weight:700;color:#4a148c;'>{next_due}</p>
              <p>Practitioner: {practitioner or 'Not recorded'}</p>
            </div>
            """, unsafe_allow_html=True)

        # ── AI CLINICAL SUMMARY — auto-generated, zero API cost ───────────────
        risk_factors_list = []
        if fam_hist != "None":              risk_factors_list.append(f"family history ({fam_hist})")
        if hrt:                             risk_factors_list.append("HRT/OCP use >5 yrs")
        if prev_biopsy:                     risk_factors_list.append("previous abnormal biopsy")
        if obesity:                         risk_factors_list.append("obesity (BMI ≥30)")
        if menopause != "Pre-menopausal":   risk_factors_list.append(menopause.lower())

        ctx = {
            "name":         p_name or "Patient",
            "age":          p_age,
            "county":       p_county,
            "facility":     p_facility,
            "risk":         risk,
            "score":        min(score, 100),
            "referral":     referral,
            "symptoms":     ", ".join(red_flags) if red_flags else "No acute red flags",
            "risk_factors": ", ".join(risk_factors_list) if risk_factors_list else "none documented",
            "ref_action":   ref_action,
        }
        st.session_state.last_assessment_context = ctx

        st.markdown("---")
        st.markdown("### 🤖 AI Clinical Summary")
        st.markdown('<div class="section-sub">Auto-generated handover note · Ready to copy into patient file</div>', unsafe_allow_html=True)

        with st.spinner("Generating clinical summary..."):
            time.sleep(0.8)   # brief pause — makes it feel like it's thinking
            summary = generate_clinical_summary(ctx)

        st.markdown(f'<div class="ai-response">{summary}</div>', unsafe_allow_html=True)
        st.caption("AI-assisted summary · Always verify with clinical judgement · Not a substitute for specialist review")

        if p_name:
            st.session_state.patients.append({
                "id":           f"PT-{len(st.session_state.patients)+1:03d}",
                "name":         p_name,
                "age":          p_age,
                "county":       p_county,
                "facility":     p_facility,
                "practitioner": practitioner,
                "risk":         risk,
                "referral":     referral,
                "next_due":     next_due,
                "status":       "Urgent" if risk == "HIGH" else "Pending",
                "notes":        ", ".join(red_flags) if red_flags else "No red flags",
            })
            st.session_state.screenings = pd.concat([
                st.session_state.screenings,
                pd.DataFrame([{
                    "date": datetime.today().strftime("%Y-%m-%d"),
                    "month": datetime.today().strftime("%b %Y"),
                    "county": p_county, "facility": p_facility,
                    "profession": "Unknown", "risk": risk,
                    "referral": referral, "followed_up": False,
                }])
            ], ignore_index=True)
            st.success(f"✅ **{p_name}** saved. Go to 🔔 Follow-Up Tracker to see her.")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 2 — SCREENING CHECKLIST
# ══════════════════════════════════════════════════════════════════════════════
elif "Screening Checklist" in module:
    st.markdown('<div class="section-head">✅ Guided Screening Checklist</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Step-by-step examination workflow · Red flags auto-detected</div>', unsafe_allow_html=True)

    steps = {
        1: ("📋 History Taking", [
            "Chief complaint documented",
            "Duration of symptoms recorded",
            "Previous breast conditions / biopsies noted",
            "Menstrual & reproductive history taken",
            "Family history (1st & 2nd degree) documented",
            "Medication history (HRT, OCP) reviewed",
            "Alcohol / smoking history noted",
        ]),
        2: ("👁️ Visual Inspection (patient seated, arms at sides)", [
            "Breast symmetry assessed — note asymmetry",
            "Skin surface checked: peau d'orange, dimpling",
            "Nipple position & symmetry checked",
            "Areola colour changes noted",
            "Visible lump or contour change observed",
            "Arms raised — repeat inspection",
            "Hands on hips (pectoral contraction) — repeat",
        ]),
        3: ("🖐️ Palpation (patient supine)", [
            "Systematic pattern chosen (spiral / radial / grid)",
            "All 4 quadrants palpated with 3-finger technique",
            "Tail of Spence (axillary extension) palpated",
            "Areola & nipple gently compressed",
            "Nipple discharge — colour & consistency noted",
            "Lump characteristics recorded: size, shape, mobility, tenderness",
        ]),
        4: ("🔍 Lymph Node Examination", [
            "Axillary nodes palpated — anterior, posterior, medial",
            "Supraclavicular nodes palpated",
            "Infraclavicular nodes palpated",
            "Node size, consistency, mobility noted",
            "Bilateral comparison performed",
        ]),
        5: ("🚩 Red Flag Detection & Decision", [
            "Hard, irregular, fixed lump present?",
            "Skin tethering or dimpling present?",
            "Bloody or spontaneous nipple discharge?",
            "Nipple retraction (new)?",
            "Axillary nodes enlarged / fixed?",
            "Ulceration or skin breakdown?",
            "Post-menopausal patient with any lump?",
        ]),
    }

    step = st.session_state.checklist_step
    col_prev, col_mid, col_next = st.columns([1, 4, 1])
    with col_prev:
        if st.button("← Back") and step > 1:
            st.session_state.checklist_step -= 1
            st.rerun()
    with col_mid:
        st.progress(step / len(steps))
        st.markdown(f"**Step {step} of {len(steps)}**")
    with col_next:
        if st.button("Next →") and step < len(steps):
            st.session_state.checklist_step += 1
            st.rerun()

    title, items = steps[step]
    st.markdown(f"### {title}")

    checked = []
    for i, item in enumerate(items):
        val = st.checkbox(item, key=f"chk_{step}_{i}")
        is_flag_step = (step == 5)
        css  = "check-item flag" if (is_flag_step and val) else "check-item"
        icon = "🚩" if (is_flag_step and val) else "✔️" if val else "○"
        st.markdown(f'<div class="{css}">{icon} {item}</div>', unsafe_allow_html=True)
        checked.append(val)

    st.markdown(f"<br>**{sum(checked)}/{len(items)} items completed**", unsafe_allow_html=True)

    tips = {
        1: "💡 Ensure patient privacy and explain each step before proceeding.",
        2: "💡 Adequate lighting is essential. Ask patient to disrobe to waist for full inspection.",
        3: "💡 Use the pads of fingers — not tips. Apply light, medium, and firm pressure at each point.",
        4: "💡 Stand on the same side as the axilla. Support the patient's arm throughout.",
        5: "💡 Red flags must be documented and acted on immediately. Never dismiss a patient concern.",
    }
    st.info(tips[step])

    if step == 5:
        rf = sum(checked)
        if rf >= 3:
            st.error(f"🚨 **{rf} red flags — URGENT REFERRAL required.**")
        elif rf >= 1:
            st.warning(f"⚠️ **{rf} red flag(s) — refer for imaging within 2 weeks.**")
        else:
            st.success("✅ No red flags — educate on BSE and schedule annual follow-up.")

    if st.button("🔄 Restart Checklist"):
        st.session_state.checklist_step = 1
        st.rerun()


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 3 — FOLLOW-UP TRACKER
# ══════════════════════════════════════════════════════════════════════════════
elif "Follow-Up" in module:
    st.markdown('<div class="section-head">🔔 Smart Follow-Up Tracker</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Track pending screenings · Flag missed follow-ups · Manage patient timelines</div>', unsafe_allow_html=True)

    df_p  = pd.DataFrame(st.session_state.patients)
    today = datetime.today().strftime("%Y-%m-%d")

    def auto_status(row):
        if row["next_due"] < today and row["status"] not in ("Done",):
            return "Overdue"
        return row["status"]
    df_p["status"] = df_p.apply(auto_status, axis=1)

    n_total   = len(df_p)
    n_overdue = len(df_p[df_p["status"] == "Overdue"])
    n_high    = len(df_p[df_p["risk"]   == "HIGH"])
    n_pending = len(df_p[df_p["status"] == "Pending"])

    t1, t2, t3, t4 = st.columns(4)
    for col, val, lbl, clr in [
        (t1, n_total,   "Total Patients",    "#6a1b9a"),
        (t2, n_overdue, "Overdue",           "#e53935"),
        (t3, n_high,    "High Risk",         "#f57c00"),
        (t4, n_pending, "Pending Follow-Up", "#1565c0"),
    ]:
        col.markdown(f"""<div class="mtile" style="border-top-color:{clr};">
          <h2 style="color:{clr};">{val}</h2><p>{lbl}</p></div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    fc1, fc2 = st.columns(2)
    f_status = fc1.selectbox("Filter by Status", ["All","Overdue","Pending","Scheduled","Urgent","Done"])
    f_risk   = fc2.selectbox("Filter by Risk",   ["All","HIGH","MODERATE","LOW"])

    view = df_p.copy()
    if f_status != "All": view = view[view["status"] == f_status]
    if f_risk   != "All": view = view[view["risk"]   == f_risk]

    st.markdown(f"**Showing {len(view)} patients**")

    for _, row in view.iterrows():
        diff    = (datetime.strptime(row["next_due"], "%Y-%m-%d") - datetime.today()).days
        due_txt = f"⚠️ {abs(diff)} days overdue" if diff < 0 else f"Due in {diff} days"
        risk_icon = "🔴" if row["risk"]=="HIGH" else "🟡" if row["risk"]=="MODERATE" else "🟢"

        with st.expander(f"**{row['id']}** · {row['name']} · {row['county']} · {risk_icon} {row['risk']} · {row['status']}"):
            ec1, ec2, ec3 = st.columns(3)
            ec1.markdown(f"**Age:** {row['age']}  \n**Facility:** {row['facility']}")
            ec2.markdown(f"**Practitioner:** {row['practitioner']}  \n**Referral:** {row['referral']}")
            ec3.markdown(f"**Next Due:** `{row['next_due']}`  \n**{due_txt}**")
            st.markdown(f"**Notes:** {row['notes']}")

            a1, a2, a3 = st.columns(3)
            if a1.button("✅ Mark Done",         key=f"done_{row['id']}"):
                for i, p in enumerate(st.session_state.patients):
                    if p["id"] == row["id"]: st.session_state.patients[i]["status"] = "Done"
                st.rerun()
            if a2.button("📅 Reschedule +2 wks", key=f"rs_{row['id']}"):
                nd = (datetime.today() + timedelta(weeks=2)).strftime("%Y-%m-%d")
                for i, p in enumerate(st.session_state.patients):
                    if p["id"] == row["id"]:
                        st.session_state.patients[i]["next_due"] = nd
                        st.session_state.patients[i]["status"]   = "Scheduled"
                st.rerun()
            if a3.button("🚨 Flag Urgent",        key=f"urg_{row['id']}"):
                for i, p in enumerate(st.session_state.patients):
                    if p["id"] == row["id"]:
                        st.session_state.patients[i]["status"] = "Urgent"
                        st.session_state.patients[i]["risk"]   = "HIGH"
                st.rerun()

    st.markdown("---")
    st.markdown("### ➕ Add Patient Manually")
    with st.form("add_pt"):
        nc1, nc2, nc3 = st.columns(3)
        np_name   = nc1.text_input("Patient Name")
        np_age    = nc1.number_input("Age", 18, 90, 35)
        np_county = nc2.selectbox("County", ["Nairobi","Mombasa","Kisumu","Kiambu","Nakuru","Uasin Gishu","Homa Bay","Other"])
        np_prac   = nc2.text_input("Practitioner")
        np_risk   = nc3.selectbox("Risk Level", ["LOW","MODERATE","HIGH"])
        np_ref    = nc3.selectbox("Referral",   ["Education","Routine","Imaging","Urgent"])
        np_notes  = st.text_area("Clinical Notes", height=68)
        np_due    = st.date_input("Next Follow-Up Date", datetime.today() + timedelta(days=14))
        if st.form_submit_button("➕ Add Patient", use_container_width=True) and np_name:
            st.session_state.patients.append({
                "id":           f"PT-{len(st.session_state.patients)+1:03d}",
                "name":         np_name, "age": np_age, "county": np_county,
                "facility":     "—", "practitioner": np_prac,
                "risk":         np_risk, "referral": np_ref,
                "next_due":     np_due.strftime("%Y-%m-%d"),
                "status":       "Scheduled", "notes": np_notes,
            })
            st.success(f"✅ {np_name} added.")
            st.rerun()


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 4 — REFERRAL INTELLIGENCE
# ══════════════════════════════════════════════════════════════════════════════
elif "Referral" in module:
    st.markdown('<div class="section-head">🔀 Referral Intelligence Engine</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Select patient symptoms · Receive evidence-based referral decision</div>', unsafe_allow_html=True)

    rc1, rc2 = st.columns(2)
    with rc1:
        st.markdown("**Clinical Signs**")
        s_lump     = st.checkbox("🔴 Palpable breast lump")
        s_axillary = st.checkbox("🔴 Axillary lymphadenopathy")
        s_skin     = st.checkbox("🟠 Skin changes / dimpling")
        s_nipple_i = st.checkbox("🟠 Nipple inversion (new)")
        s_nipple_d = st.checkbox("🟠 Nipple discharge (non-milk)")
        s_ulcer    = st.checkbox("🔴 Ulceration / wound on breast")
        s_pain     = st.checkbox("🟡 Breast pain (persistent)")
        s_asymm    = st.checkbox("🟡 Breast asymmetry (new)")
    with rc2:
        st.markdown("**Patient Context**")
        age_grp   = st.selectbox("Age group", ["<35 years","35–49 years","50+ years"])
        fhx       = st.selectbox("Family history", ["None","1st-degree relative","Strong / BRCA"])
        prev_ca   = st.checkbox("Personal history of any cancer")
        post_meno = st.checkbox("Post-menopausal")
        rapid_chg = st.checkbox("Rapid change in size / appearance")
        bilateral = st.checkbox("Bilateral symptoms")

    if st.button("🔀  Generate Referral Decision", use_container_width=True):
        u, img, r, e = 0, 0, 0, 0
        if s_lump:     u += 3; img += 2
        if s_axillary: u += 4
        if s_ulcer:    u += 5
        if s_skin:     u += 3; img += 2
        if s_nipple_i: u += 2; img += 2
        if s_nipple_d: u += 1; img += 3
        if rapid_chg:  u += 2
        if prev_ca:    u += 3
        if fhx == "Strong / BRCA":         u += 2; img += 2
        elif fhx == "1st-degree relative": img += 2; r += 1
        if post_meno and s_lump: u += 2
        if age_grp == "50+ years": img += 1
        if s_pain and not s_lump:  r += 2; e += 1
        if s_asymm and not s_lump: img += 1; r += 1
        if not any([s_lump, s_axillary, s_skin, s_nipple_i, s_nipple_d, s_ulcer]): e += 3

        scores   = {"Urgent": u, "Imaging": img, "Routine": r, "Education": e}
        decision = max(scores, key=scores.get)

        ref_details = {
            "Urgent":    ("🚨","#e53935","Refer within 24 hours to oncology/surgical unit.",
                          ["Contact receiving facility before sending","Provide written referral letter",
                           "Ensure patient has transport","Document and notify facility in-charge"]),
            "Imaging":   ("🖼️","#1565c0","Mammogram and/or ultrasound within 2 weeks.",
                          ["Bilateral mammogram if ≥40 years","Ultrasound preferred if <40 or dense breasts",
                           "Provide patient with written imaging request","Book follow-up to review results"]),
            "Routine":   ("📅","#f57c00","Reassess within 4–6 weeks.",
                          ["Educate patient on breast self-examination","Document current findings",
                           "Book firm follow-up before patient leaves","Advise to return if symptoms worsen"]),
            "Education": ("📚","#2e7d32","No acute clinical concern identified.",
                          ["Demonstrate BSE technique","Provide take-home BSE card",
                           "Discuss annual screening schedule","Advise on risk reduction"]),
        }
        icon, clr, headline, actions = ref_details[decision]
        actions_html = "".join([f"<li style='margin:.4rem 0;'>{a}</li>" for a in actions])

        st.markdown("---")
        st.markdown(f"""
        <div style='background:{clr};color:white;border-radius:16px;padding:1.5rem 2rem;margin-bottom:1rem;
                    font-family:Nunito,sans-serif;'>
          <div style='font-size:2rem;'>{icon}</div>
          <div style='font-size:1.45rem;font-weight:800;margin:.25rem 0;'>{decision} Referral</div>
          <div style='font-size:.97rem;opacity:.92;'>{headline}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="kcard">
          <h3>📋 Action Steps</h3>
          <ul style='margin:.5rem 0;padding-left:1.3rem;color:#333;font-size:.93rem;
                     font-family:Nunito,sans-serif;'>{actions_html}</ul>
        </div>
        """, unsafe_allow_html=True)

        fig_s = px.bar(x=list(scores.keys()), y=list(scores.values()),
                       color=list(scores.keys()),
                       color_discrete_map={"Urgent":"#e53935","Imaging":"#1565c0",
                                           "Routine":"#f57c00","Education":"#2e7d32"},
                       template="plotly_white",
                       labels={"x":"Decision","y":"Evidence Score"},
                       title="Evidence weighting behind recommendation")
        fig_s.update_layout(showlegend=False, font_family="Nunito")
        st.plotly_chart(fig_s, use_container_width=True)

        if decision in ["Urgent","Imaging"]:
            st.warning("⚠️ **Practice gap alert:** Kenya KAP data shows only 12–20% of cases meeting urgent criteria are referred. This patient meets the threshold — please act.")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 5 — ANALYTICS DASHBOARD
# ══════════════════════════════════════════════════════════════════════════════
elif "Analytics" in module:
    st.markdown('<div class="section-head">📊 Administrator Analytics Dashboard</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Screenings performed · Referral patterns · Follow-up compliance · County & facility breakdown</div>', unsafe_allow_html=True)

    df_s = st.session_state.screenings.copy()
    df_p = pd.DataFrame(st.session_state.patients)

    fc1, fc2 = st.columns(2)
    cf = fc1.selectbox("County",   ["All"] + sorted(df_s["county"].unique()))
    ff = fc2.selectbox("Facility", ["All"] + sorted(df_s["facility"].unique()))
    if cf != "All": df_s = df_s[df_s["county"]   == cf]
    if ff != "All": df_s = df_s[df_s["facility"] == ff]

    n_screen    = len(df_s)
    n_urgent    = len(df_s[df_s["risk"] == "HIGH"])
    pct_fu      = round(df_s["followed_up"].mean() * 100, 1)
    n_overdue_p = len(df_p[df_p["status"] == "Overdue"])

    t1, t2, t3, t4 = st.columns(4)
    for col, val, lbl, clr in [
        (t1, n_screen,     "Screenings (90 days)", "#6a1b9a"),
        (t2, n_urgent,     "High-Risk Cases",      "#e53935"),
        (t3, f"{pct_fu}%", "Follow-Up Rate",       "#f57c00"),
        (t4, n_overdue_p,  "Overdue Follow-Ups",   "#1565c0"),
    ]:
        col.markdown(f"""<div class="mtile" style="border-top-color:{clr};">
          <h2 style="color:{clr};">{val}</h2><p>{lbl}</p></div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    month_order = sorted(df_s["month"].unique(), key=lambda m: datetime.strptime(m, "%b %Y"))
    trend = df_s.groupby("month").size().reset_index(name="Screenings")
    trend["month"] = pd.Categorical(trend["month"], categories=month_order, ordered=True)
    trend = trend.sort_values("month")
    fig_trend = px.line(trend, x="month", y="Screenings", markers=True,
                        template="plotly_white", title="📈 Screenings Over Time",
                        color_discrete_sequence=["#6a1b9a"])
    fig_trend.update_traces(line_width=3, marker_size=8)
    fig_trend.update_layout(font_family="Nunito")
    st.plotly_chart(fig_trend, use_container_width=True)

    cl, cr = st.columns(2)
    with cl:
        cnt = df_s.groupby("county").size().reset_index(name="Screenings").sort_values("Screenings")
        fig = px.bar(cnt, x="Screenings", y="county", orientation="h",
                     color_discrete_sequence=["#7b1fa2"],
                     template="plotly_white", title="🗺️ Screenings by County")
        fig.update_layout(font_family="Nunito")
        st.plotly_chart(fig, use_container_width=True)
    with cr:
        rc = df_s["referral"].value_counts().reset_index()
        rc.columns = ["Referral","Count"]
        fig = px.pie(rc, names="Referral", values="Count",
                     color="Referral",
                     color_discrete_map={"Urgent":"#e53935","Imaging":"#1565c0",
                                         "Routine":"#f57c00","Education":"#2e7d32"},
                     template="plotly_white", title="🔀 Referral Breakdown")
        fig.update_layout(font_family="Nunito")
        st.plotly_chart(fig, use_container_width=True)

    cl2, cr2 = st.columns(2)
    with cl2:
        fc = df_s.groupby("facility").size().reset_index(name="Screenings")
        fig = px.bar(fc, x="facility", y="Screenings",
                     color_discrete_sequence=["#1565c0"],
                     template="plotly_white", title="🏥 Screenings by Facility")
        fig.update_layout(xaxis_tickangle=-20, font_family="Nunito")
        st.plotly_chart(fig, use_container_width=True)
    with cr2:
        fu = df_s.groupby("county")["followed_up"].mean().mul(100).round(1).reset_index()
        fu.columns = ["County","Follow-Up Rate (%)"]
        fig = px.bar(fu.sort_values("Follow-Up Rate (%)"),
                     x="Follow-Up Rate (%)", y="County", orientation="h",
                     color="Follow-Up Rate (%)",
                     color_continuous_scale=["#e53935","#f57c00","#2e7d32"],
                     template="plotly_white", title="📋 Follow-Up Compliance by County")
        fig.add_vline(x=pct_fu, line_dash="dash", line_color="#6a1b9a", annotation_text="Avg")
        fig.update_layout(font_family="Nunito")
        st.plotly_chart(fig, use_container_width=True)

    rp = df_s.groupby(["profession","risk"]).size().reset_index(name="Count")
    fig = px.bar(rp, x="profession", y="Count", color="risk",
                 color_discrete_map={"HIGH":"#e53935","MODERATE":"#f57c00","LOW":"#2e7d32"},
                 template="plotly_white", barmode="stack",
                 title="⚕️ Risk Distribution by Profession")
    fig.update_layout(xaxis_tickangle=-20, font_family="Nunito")
    st.plotly_chart(fig, use_container_width=True)

    overdue = df_p[df_p["status"] == "Overdue"].sort_values("next_due")
    if not overdue.empty:
        st.markdown("### 🚨 Overdue Follow-Ups")
        st.error(f"**{len(overdue)} patient(s) have missed their follow-up.** Immediate action required.")
        st.dataframe(
            overdue[["id","name","county","facility","risk","referral","next_due","practitioner","notes"]]
            .reset_index(drop=True),
            use_container_width=True
        )
    else:
        st.success("✅ No overdue follow-ups at this time.")

    st.markdown("---")
    csv = df_s.to_csv(index=False).encode("utf-8")
    st.download_button("⬇️ Export Screening Data (CSV)", data=csv,
                       file_name="breastcare_kenya_screenings.csv", mime="text/csv")


# ══════════════════════════════════════════════════════════════════════════════
# MODULE 6 — AI CLINICAL ASSISTANT
# ══════════════════════════════════════════════════════════════════════════════
elif "AI Clinical" in module:
    st.markdown('<div class="section-head">🤖 AI Clinical Assistant</div>', unsafe_allow_html=True)
    st.markdown('<div class="section-sub">Ask clinical questions · Generate patient education · Draft referral letters · Interpret analytics</div>', unsafe_allow_html=True)

    st.markdown("""
    <div class="ai-card">
      <h3>What can I help with?</h3>
      <p>Clinical questions about breast cancer signs, symptoms & risk factors ·
      BSE instructions in Swahili · Referral letter generation · Analytics interpretation ·
      Kenya-specific referral pathways · WHO screening guidelines</p>
    </div>
    """, unsafe_allow_html=True)

    # ── Quick-action buttons ──────────────────────────────────────────────────
    st.markdown("**Quick Actions**")
    qa_cols = st.columns(4)
    quick_actions = [
        ("📋 Summarise last patient", "summarise last patient assessed — give a clinical handover note"),
        ("📝 Draft referral letter",  "draft a referral letter for the last patient assessed"),
        ("🇰🇪 BSE in Swahili",         "explain BSE in Swahili"),
        ("📊 Interpret analytics",    "explain the analytics and screening statistics for a county health officer"),
    ]
    for col, (label, prompt) in zip(qa_cols, quick_actions):
        if col.button(label, use_container_width=True, key=f"qa_{label}"):
            st.session_state.ai_chat_history.append({"role": "user", "content": prompt})
            st.rerun()

    st.markdown("---")
    st.markdown("### 💬 Clinical Q&A")

    # ── Render chat history ───────────────────────────────────────────────────
    for msg in st.session_state.ai_chat_history:
        if msg["role"] == "user":
            st.markdown(f'<div class="chat-user">👤 {msg["content"]}</div>', unsafe_allow_html=True)
        else:
            st.markdown(f'<div class="chat-ai">🤖 {msg["content"]}</div>', unsafe_allow_html=True)

    # ── Auto-respond if last message is from user ─────────────────────────────
    if st.session_state.ai_chat_history and st.session_state.ai_chat_history[-1]["role"] == "user":
        last_user_msg = st.session_state.ai_chat_history[-1]["content"]
        with st.spinner("Thinking..."):
            time.sleep(0.6)
            response = ai_knowledge_base(last_user_msg, st.session_state.last_assessment_context)
        st.session_state.ai_chat_history.append({"role": "assistant", "content": response})
        st.rerun()

    # ── Input form ────────────────────────────────────────────────────────────
    with st.form("chat_form", clear_on_submit=True):
        user_input = st.text_input(
            "Ask a clinical question...",
            placeholder="e.g. What does peau d'orange mean? / How do I explain a referral to a patient? / What are WHO guidelines?"
        )
        send_col, clear_col = st.columns([4, 1])
        send  = send_col.form_submit_button("Send →", use_container_width=True)
        clear = clear_col.form_submit_button("🗑️ Clear", use_container_width=True)

    if send and user_input.strip():
        st.session_state.ai_chat_history.append({"role": "user", "content": user_input.strip()})
        st.rerun()

    if clear:
        st.session_state.ai_chat_history = []
        st.rerun()

    st.caption("⚠️ AI responses are for clinical support only. Always verify with clinical judgement. Not a substitute for specialist review.")

2026-05-28 15:14:46.682 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 15:14:46.692 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 15:14:47.994 
  command:

    streamlit run C:\Users\Naomi Bosire\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-28 15:14:47.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 15:14:47.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 15:14:47.994 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-28 15:14:47.994 Session state does not function when running a script with